In [26]:
import json
import pandas as pd
import os
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_ollama import OllamaEmbeddings, OllamaLLM
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from ragas import evaluate, RunConfig
from ragas import EvaluationDataset
from ragas.dataset_schema import EvaluationResult
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness

In [27]:
def load_data(file_path):
    with open(file_path, 'r') as file:
        data = json.load(file)
    return data

In [28]:
train_file = "train_short.json"
train = load_data(train_file)

In [29]:
# convert json to dataframe
train_df = pd.DataFrame(train)

In [30]:
# convert json to dataframe
train_df = pd.DataFrame(train)

# traindf where wellformedanswers is not empty
train_df.drop(columns=["query_id", "query_type", "wellFormedAnswers"], inplace=True)

# for answer in column answers apply lambda function to get the first element
train_df["answers"] = train_df["answers"].apply(lambda x: x[0] if len(x) > 0 else None)

# split passages into twenty columns (passage_0_text, passage_0_is_selected, passage_1_text, passage_1_is_selected, ...)
for i in range(10):
    passage_text = train_df["passages"].apply(lambda x: x[i]["passage_text"])
    passage_is_selected = train_df["passages"].apply(lambda x: x[i]["is_selected"])
    train_df[f"passage_{i}_selected"] = passage_is_selected
    train_df[f"passage_{i}_text"] = passage_text

# drop columns passage
train_df.drop(columns=["passages"], inplace=True)

# train_df = train_df.iloc[:10]
train_df = train_df[:10]
train_df

,answers,query,passage_0_selected,passage_0_text,passage_1_selected,passage_1_text,passage_2_selected,passage_2_text,passage_3_selected,passage_3_text,...,passage_5_selected,passage_5_text,passage_6_selected,passage_6_text,passage_7_selected,passage_7_text,passage_8_selected,passage_8_text,passage_9_selected,passage_9_text
0,The immediate impact of the success of the man...,)what was the immediate impact of the success ...,1,The presence of communication amid scientific ...,0,The Manhattan Project and its atomic bomb help...,0,Essay on The Manhattan Project - The Manhattan...,0,The Manhattan Project was the name for a proje...,...,0,The Manhattan Project. This once classified ph...,0,Nor will it attempt to substitute for the extr...,0,Manhattan Project. The Manhattan Project was a...,0,"In June 1942, the United States Army Corps of ...",0,One of the main reasons Hanford was selected a...
1,Restorative justice that fosters dialogue betw...,_________ justice is designed to repair the ha...,0,"group discussions, community boards or panels ...",0,punishment designed to repair the damage done ...,0,Tutorial: Introduction to Restorative Justice....,0,"Organize volunteer community panels, boards, o...",...,0,Each of these types of communities—the geograp...,1,The approach is based on a theory of justice t...,0,Inherent in many people’s understanding of the...,0,"Criminal justice, however, is not usually conc...",0,The circle includes a wide range of participan...
2,The customer care number of Amex India is 1800...,amex india customer care number,0,American Express is the world's premier servic...,0,Customer Service Main Page: Get Information: U...,1,Amex Card India Customer Care Phone Number Pho...,0,Air India American Express Gold Card Customer ...,...,0,"Update Address, Payee Name, Bank Account and C...",0,Please mention the first 11 digits of your Ame...,0,"The postal and official address, email address...",0,Corporate Cardmembers can contact the 24 hour ...,0,Through which any customers can easily contact...
3,"Ramen is a quick-cooking noodles, typically se...",definition of ramen,0,"Ramen is of Chinese origin, however it is uncl...",0,"ramen definition, meaning, what is ramen: a Ja...",0,Definition of ramen written for English Langua...,0,Sapporo ramen comes from Japan's northernmost ...,...,0,Wiktionary (0.00 / 0 votes) Rate this definiti...,0,Related dishes. 1 Nagasaki champon. The noodl...,0,[sometimes with sing. v.] Japanese noodles of ...,0,"‘The noodles, most of which we left behind bec...",0,"On the ground floor level, there is a souvenir..."
4,Rachel Carson died because of cancer.,why did rachel carson die,0,A Fish and Wildlife Service National Wildlife ...,0,Though much of their correspondence was destro...,0,The impetus for Silent Spring was a letter wri...,0,How about the environmentalist and writer Rach...,...,0,"Rachel Carson was born on May 27, 1907, on a f...",0,Lived 1907 – 1964. Rachel Carson played a key ...,0,Rachel Carson had a large love for nature and ...,1,Heroes commit feats of courage involving risk ...,0,"To passers-by the mother would say, pointing, ..."
5,Rosa parks protested because African Americans...,why did rosa parks protest,0,"Rosa Parks, an African American, was arrested ...",0,Rosa Parks and the Montgomery Bus Boycott. On ...,0,Sparked by the arrest of Rosa Parks on 1 Decem...,0,After the Boycott. Rosa Parks and her husband ...,...,0,(1913-2005). By refusing to give up her bus se...,1,Why did Rosa parks protest about the black rig...,0,"In Montgomery, Alabama, Rosa Parks is jailed f...",0,"Civil rights activist Rosa Parks (February 4, ...",0,"Legacy and honors. 1 1976, Detroit renamed 12..."
6,The sabertooth tiger may have become extinct d...,why did saber toothed tigers went extinct,0,The Saber Tooth Tiger is also called the saber...,0,When in history did the saber-toothed tiger be...,0,Recently conducted studies on the fossils of s...,0,"Sabre-toothed cat: Sabre-toothed cat, any of t...",..

In [31]:
sample_queries = list(train_df['query'])
expected_responses = list(train_df['answers'])

# create a list of every passage, also append the rowid stretched to 4 digits + passage_0_selected number
passages = []
for i, row in train_df.iterrows():
    for j in range(10):
        passage = row[f"passage_{j}_text"]
        passage_is_selected = row[f"passage_{j}_selected"]
        id = f"{int(i):04d}_{passage_is_selected}"
        passages.append((id, passage))

In [32]:
dataset_documents = [
    Document(
        page_content=one_passage,
        metadata={
            "id": one_id,
        }
    ) for one_id, one_passage in passages
]

In [33]:
embeddings = OllamaEmbeddings(model="mxbai-embed-large")
dataset_vector_store = InMemoryVectorStore(embeddings)

dataset_vector_store.add_documents(dataset_documents)
dataset_retriever = dataset_vector_store.as_retriever(search_kwargs={"k": 4})

In [34]:
llm = OllamaLLM(model="gemma3:4b", temperature=0.0, max_tokens=256)

In [ ]:
template = """
Answer the question based ONLY on the following context (some documents may be irrelevant, select relevant sources and use them to answer the question).
If the context does not contain the answer, say "No answer provided". Answer briefly, but include all the relevant information if it was directly asked.

Context:
{context}


Question: {query}
"""
prompt = ChatPromptTemplate.from_template(template)

qa_chain = prompt | llm | StrOutputParser()

In [36]:
def build_evaluation_dataset(queries, references, retriever, qa_chain, formatter):
    data = []
    for query, reference in zip(queries, references):
        relevant_docs = retriever.invoke(query)
        response = qa_chain.invoke({"context": formatter(relevant_docs), "query": query})
        data.append({
            "user_input": query,
            "retrieved_contexts": [f"Document id {doc.metadata["id"]}:\n{doc.page_content}" for doc in relevant_docs],
            "response": response,
            "reference": reference,
        })
    return EvaluationDataset.from_list(data)


def format_docs(relevant_docs):
    return [f"Document:\n{doc.page_content}\n\n" for doc in relevant_docs]

In [37]:
# train_df = train_df.loc[["11"]]
train_df

,answers,query,passage_0_selected,passage_0_text,passage_1_selected,passage_1_text,passage_2_selected,passage_2_text,passage_3_selected,passage_3_text,...,passage_5_selected,passage_5_text,passage_6_selected,passage_6_text,passage_7_selected,passage_7_text,passage_8_selected,passage_8_text,passage_9_selected,passage_9_text
0,The immediate impact of the success of the man...,)what was the immediate impact of the success ...,1,The presence of communication amid scientific ...,0,The Manhattan Project and its atomic bomb help...,0,Essay on The Manhattan Project - The Manhattan...,0,The Manhattan Project was the name for a proje...,...,0,The Manhattan Project. This once classified ph...,0,Nor will it attempt to substitute for the extr...,0,Manhattan Project. The Manhattan Project was a...,0,"In June 1942, the United States Army Corps of ...",0,One of the main reasons Hanford was selected a...
1,Restorative justice that fosters dialogue betw...,_________ justice is designed to repair the ha...,0,"group discussions, community boards or panels ...",0,punishment designed to repair the damage done ...,0,Tutorial: Introduction to Restorative Justice....,0,"Organize volunteer community panels, boards, o...",...,0,Each of these types of communities—the geograp...,1,The approach is based on a theory of justice t...,0,Inherent in many people’s understanding of the...,0,"Criminal justice, however, is not usually conc...",0,The circle includes a wide range of participan...
2,The customer care number of Amex India is 1800...,amex india customer care number,0,American Express is the world's premier servic...,0,Customer Service Main Page: Get Information: U...,1,Amex Card India Customer Care Phone Number Pho...,0,Air India American Express Gold Card Customer ...,...,0,"Update Address, Payee Name, Bank Account and C...",0,Please mention the first 11 digits of your Ame...,0,"The postal and official address, email address...",0,Corporate Cardmembers can contact the 24 hour ...,0,Through which any customers can easily contact...
3,"Ramen is a quick-cooking noodles, typically se...",definition of ramen,0,"Ramen is of Chinese origin, however it is uncl...",0,"ramen definition, meaning, what is ramen: a Ja...",0,Definition of ramen written for English Langua...,0,Sapporo ramen comes from Japan's northernmost ...,...,0,Wiktionary (0.00 / 0 votes) Rate this definiti...,0,Related dishes. 1 Nagasaki champon. The noodl...,0,[sometimes with sing. v.] Japanese noodles of ...,0,"‘The noodles, most of which we left behind bec...",0,"On the ground floor level, there is a souvenir..."
4,Rachel Carson died because of cancer.,why did rachel carson die,0,A Fish and Wildlife Service National Wildlife ...,0,Though much of their correspondence was destro...,0,The impetus for Silent Spring was a letter wri...,0,How about the environmentalist and writer Rach...,...,0,"Rachel Carson was born on May 27, 1907, on a f...",0,Lived 1907 – 1964. Rachel Carson played a key ...,0,Rachel Carson had a large love for nature and ...,1,Heroes commit feats of courage involving risk ...,0,"To passers-by the mother would say, pointing, ..."
5,Rosa parks protested because African Americans...,why did rosa parks protest,0,"Rosa Parks, an African American, was arrested ...",0,Rosa Parks and the Montgomery Bus Boycott. On ...,0,Sparked by the arrest of Rosa Parks on 1 Decem...,0,After the Boycott. Rosa Parks and her husband ...,...,0,(1913-2005). By refusing to give up her bus se...,1,Why did Rosa parks protest about the black rig...,0,"In Montgomery, Alabama, Rosa Parks is jailed f...",0,"Civil rights activist Rosa Parks (February 4, ...",0,"Legacy and honors. 1 1976, Detroit renamed 12..."
6,The sabertooth tiger may have become extinct d...,why did saber toothed tigers went extinct,0,The Saber Tooth Tiger is also called the saber...,0,When in history did the saber-toothed tiger be...,0,Recently conducted studies on the fossils of s...,0,"Sabre-toothed cat: Sabre-toothed cat, any of t...",..

In [38]:
evaluation_dataset = build_evaluation_dataset(
    list(train_df['query']),
    list(train_df['answers']),
    dataset_retriever,
    qa_chain,
    format_docs
)

In [39]:
evaluation_dataset_filtered = EvaluationDataset(list(filter(lambda x: x.reference != "No Answer Present.", evaluation_dataset.samples)))

In [40]:
len(evaluation_dataset_filtered)

9

In [41]:
for sample in evaluation_dataset.samples:
    print(f"Query: {sample.user_input}")
    print(f"LLM answer: {sample.response}")
    print(f"Reference answer: {sample.reference}")
    print()

Query: )what was the immediate impact of the success of the manhattan project?
LLM answer: The success of the Manhattan Project helped bring an end to World War II.
Reference answer: The immediate impact of the success of the manhattan project was the only cloud hanging over the impressive achievement of the atomic researchers and engineers is what their success truly meant; hundreds of thousands of innocent lives obliterated.

Query: _________ justice is designed to repair the harm to victim, the community and the offender caused by the offender criminal act. question 19 options:
LLM answer: punishment designed to repair the damage done to the victim and community by an offender's criminal act.
Reference answer: Restorative justice that fosters dialogue between victim and offender has shown the highest rates of victim satisfaction and offender accountability.

Query: amex india customer care number
LLM answer: 18001801030 / 1800419103
Reference answer: The customer care number of Amex

In [42]:
run_config = RunConfig(timeout=60*10, max_workers=1)
evaluator_llm = LangchainLLMWrapper(llm)

In [43]:
result_correctness: EvaluationResult = evaluate(
    dataset=evaluation_dataset,
    metrics=[FactualCorrectness()],
    llm=evaluator_llm,
    run_config=run_config,
)

result_correctness

Evaluating: 100%|██████████| 10/10 [06:48<00:00, 40.86s/it]


{'factual_correctness(mode=f1)': 0.6330}

In [18]:
result_faithfulness: EvaluationResult = evaluate(
    dataset=evaluation_dataset_filtered,
    metrics=[Faithfulness()],
    llm=evaluator_llm,
    run_config=run_config,
)

result_faithfulness

Evaluating: 100%|██████████| 9/9 [06:05<00:00, 40.62s/it]


{'faithfulness': 0.9656}

In [19]:
print(evaluation_dataset.samples[0].model_dump()['response'])

The success of the Manhattan Project helped bring an end to World War II and forever changed the world due to the creation of an atomic bomb.


In [20]:
print(evaluation_dataset.samples[0].model_dump()['reference'])

The immediate impact of the success of the manhattan project was the only cloud hanging over the impressive achievement of the atomic researchers and engineers is what their success truly meant; hundreds of thousands of innocent lives obliterated.


In [21]:
assert False, "STOP HERE"

AssertionError: STOP HERE

---
---

In [ ]:
content_list = [
    "Andrew Ng is the CEO of Landing AI and is known for his pioneering work in deep learning. He is also widely recognized for democratizing AI education through platforms like Coursera.",
    "Sam Altman is the CEO of OpenAI and has played a key role in advancing AI research and development. He is a strong advocate for creating safe and beneficial AI technologies.",
    "Demis Hassabis is the CEO of DeepMind and is celebrated for his innovative approach to artificial intelligence. He gained prominence for developing systems that can master complex games like AlphaGo.",
    "Sundar Pichai is the CEO of Google and Alphabet Inc., and he is praised for leading innovation across Google's vast product ecosystem. His leadership has significantly enhanced user experiences on a global scale.",
    "Arvind Krishna is the CEO of IBM and is recognized for transforming the company towards cloud computing and AI solutions. He focuses on providing cutting-edge technologies to address modern business challenges.",
]

langchain_documents = []

for content in content_list:
    langchain_documents.append(
        Document(
            page_content=content,
        )
    )

In [ ]:
vector_store = InMemoryVectorStore(embeddings)

vector_store.add_documents(langchain_documents)

['064734f1-5c52-4f5a-bb36-ef5711f58d05',
 'e8816233-a3be-4580-81e6-480571274be0',
 '3efd854e-a87c-489f-9eb2-53dbe9de40da',
 '670422cc-e0f8-452b-8fcc-374f6220f291',
 '9f960f68-0633-4947-8135-598b4945db04']

In [ ]:
retriever = vector_store.as_retriever(search_kwargs={"k": 1})

In [ ]:
llm = OllamaLLM(model="gemma3:4b", temperature=0.0, max_tokens=512)

In [ ]:
template = """Answer the question based only on the following context:
{context}

Question: {query}
"""
prompt = ChatPromptTemplate.from_template(template)

qa_chain = prompt | llm | StrOutputParser()

In [ ]:
query = "Who is the CEO of OpenAI?"

relevant_docs = retriever.invoke(query)
qa_chain.invoke({"context": format_docs(relevant_docs), "query": query})

'Sam Altman is the CEO of OpenAI.'

In [ ]:
sample_queries = [
    "Which CEO is widely recognized for democratizing AI education through platforms like Coursera?",
    "Who is Sam Altman?",
    "Who is Demis Hassabis and how did he gained prominence?",
    "Who is the CEO of Google and Alphabet Inc., praised for leading innovation across Google's product ecosystem?",
    "How did Arvind Krishna transformed IBM?",
]

expected_responses = [
    "Andrew Ng is the CEO of Landing AI and is widely recognized for democratizing AI education through platforms like Coursera.",
    "Sam Altman is the CEO of OpenAI and has played a key role in advancing AI research and development. He strongly advocates for creating safe and beneficial AI technologies.",
    "Demis Hassabis is the CEO of DeepMind and is celebrated for his innovative approach to artificial intelligence. He gained prominence for developing systems like AlphaGo that can master complex games.",
    "Sundar Pichai is the CEO of Google and Alphabet Inc., praised for leading innovation across Google's vast product ecosystem. His leadership has significantly enhanced user experiences globally.",
    "Arvind Krishna is the CEO of IBM and has transformed the company towards cloud computing and AI solutions. He focuses on delivering cutting-edge technologies to address modern business challenges.",
]

In [ ]:
evaluation_dataset = build_evaluation_dataset(
    sample_queries,
    expected_responses,
    retriever,
    qa_chain,
    format_docs
)

c:\Users\piotr\Desktop\cruise-screening\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
evaluator_llm = LangchainLLMWrapper(llm)

result: EvaluationResult = evaluate(
    dataset=evaluation_dataset,
    metrics=[LLMContextRecall(), Faithfulness(), FactualCorrectness()],
    llm=evaluator_llm,
)

result

Evaluating:  80%|████████  | 12/15 [02:52<00:51, 17.11s/it]Exception raised in Job[2]: TimeoutError()
Exception raised in Job[11]: TimeoutError()
Exception raised in Job[14]: TimeoutError()
Evaluating: 100%|██████████| 15/15 [03:00<00:00, 12.00s/it]


{'context_recall': 1.0000, 'faithfulness': 1.0000, 'factual_correctness(mode=f1)': 1.0000}

In [ ]:
dataset

[{'user_input': 'Which CEO is widely recognized for democratizing AI education through platforms like Coursera?',
  'retrieved_contexts': ['Andrew Ng is the CEO of Landing AI and is known for his pioneering work in deep learning. He is also widely recognized for democratizing AI education through platforms like Coursera.'],
  'response': 'Andrew Ng is widely recognized for democratizing AI education through platforms like Coursera.',
  'reference': 'Andrew Ng is the CEO of Landing AI and is widely recognized for democratizing AI education through platforms like Coursera.'},
 {'user_input': 'Who is Sam Altman?',
  'retrieved_contexts': ['Sam Altman is the CEO of OpenAI and has played a key role in advancing AI research and development. He is a strong advocate for creating safe and beneficial AI technologies.'],
  'response': 'Sam Altman is the CEO of OpenAI and has played a key role in advancing AI research and development. He is a strong advocate for creating safe and beneficial AI tec